# Notebook 03 — H.BLSTM Reproduction

**Purpose:** Reproduce the H.BLSTM model from Kalra et al. (2024) and run the
walk-forward test on SPX (then NIFTY, FTSE) for both univariate and multivariate.

Sections:
1. Bootstrap + config
2. Model architecture overview
3. Phase 0 — initial batch training
4. Phase 1+2 — walk-forward test with per-bar incremental updates
5. Predictions visualisation
6. Save per-bar predictions (needed for Diebold-Mariano test in NB04)
7. Loop over all 3 indices × 2 feature sets × 3 seeds


## 0. Bootstrap

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_URL = "https://github.com/stephugm/incremental-learning.git"
FORCE_DOWNLOAD = False

def detect_env():
    if "google.colab" in sys.modules: return "colab"
    if os.path.exists("/kaggle/input"): return "kaggle"
    return "local"

ENV = detect_env()
print(f"Environment: {ENV}")

if ENV in ("colab", "kaggle"):
    REPO = (Path("/content/incremental-learning") if ENV == "colab"
            else Path("/kaggle/working/incremental-learning"))
    if not REPO.exists():
        subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)

    # ── Step 1: install requirements (tvDatafeed sudah dipisah) ──────────
    subprocess.run(["pip", "install", "-q", "-r",
                    str(REPO / "requirements.txt")], check=False)

    # ── Step 2: tvDatafeed best-effort dari GitHub ────────────────────────
    print("Installing tvDatafeed from GitHub ...")
    try:
        r = subprocess.run(
            ["pip", "install", "-q",
             "git+https://github.com/rongardF/tvdatafeed.git"],
            capture_output=True, timeout=120,
        )
        from tvDatafeed import TvDatafeed  # noqa: F401
        print("tvDatafeed : OK  -> target 6000 bar dapat dicapai")
    except Exception as _e:
        print(f"tvDatafeed : TIDAK TERSEDIA ({_e})")
        print("  -> fallback yfinance, hanya ~1500 bar (limit 60 hari)")
        print("  -> n_bars=6000 di config TIDAK akan tercapai via yfinance")

else:
    REPO = Path().resolve()
    while not (REPO / "configs" / "config.yaml").exists() and REPO != REPO.parent:
        REPO = REPO.parent

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print(f"Repo root: {REPO}")
# autoreload: best-effort, skip gracefully on Python 3.12+ if imp removed
try:
    _ip = get_ipython()  # noqa: F821
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
except Exception as _ar_e:
    print(f'autoreload skipped: {_ar_e}')

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.paths import Paths
from src.data import build_dataset
from src.model import BiLSTMModel, get_device
from src.hblstm import HBLSTMRunner
from src.metrics import evaluate, compare_to_paper

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

paths = Paths()
with open(paths.configs / "config.yaml") as f:
    cfg = yaml.safe_load(f)

DEVICE = get_device(cfg["runtime"]["device"])
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

INDICES      = cfg["data"]["indices"]
FEATURE_SETS = list(cfg["data"]["feature_sets"].keys())
LAG          = cfg["data"]["lag"]
TRAIN_RATIO  = cfg["data"]["split"]["train"]
N_BARS       = cfg["data"]["n_bars"]
EMA5_PERIOD  = cfg["indicators"]["ema5_period"]
SEEDS        = cfg["training"]["seeds"]

MODEL_KWARGS = {
    "hidden_size": cfg["model"]["hidden_size"],
    "num_layers":  cfg["model"]["num_layers"],
    "dropout":     cfg["model"]["dropout"],
}
TRAINER_KWARGS = {
    "lr":         cfg["training"]["learning_rate"],
    "max_epochs": cfg["training"]["max_epochs"],
    "batch_size": cfg["training"]["batch_size"],
    "verbose":    True,
}

print(f"Model: {MODEL_KWARGS}")
print(f"Training: {TRAINER_KWARGS}")
print(f"Seeds: {SEEDS}")

## 1. Quick SPX demo (single seed, multivariate)

Run this section first to verify the reproduction works before the full loop.

In [ ]:
DEMO_SYMBOL   = "SPX"
DEMO_EXCHANGE = "NYSE"
DEMO_TZ       = "America/New_York"
DEMO_FSET     = "multivariate"
DEMO_SEED     = SEEDS[0]

print(f"Demo: {DEMO_SYMBOL} | {DEMO_FSET} | seed={DEMO_SEED}")

In [ ]:
# Build dataset (loads from cache if already processed)
ds = build_dataset(
    symbol=DEMO_SYMBOL, exchange=DEMO_EXCHANGE, tz=DEMO_TZ,
    feature_set=DEMO_FSET, paths=paths,
    n_bars=N_BARS, lag=LAG, train_ratio=TRAIN_RATIO,
    force_download=False, ema5_period=EMA5_PERIOD,
)

print(f"X_train: {ds['X_train'].shape}  y_train: {ds['y_train'].shape}")
print(f"X_test:  {ds['X_test'].shape}   y_test:  {ds['y_test'].shape}")

# Extract session_ids aligned with the test set
df_clean = ds["df_clean"]
# session_ids for test timestamps
session_ids_test = np.array(
    df_clean.loc[ds["ts_test"], "session_id"].values
)

In [ ]:
# Phase 0 — initial batch training
runner = HBLSTMRunner(
    model_kwargs=MODEL_KWARGS,
    trainer_kwargs=TRAINER_KWARGS,
    incremental_lr=cfg["training"]["incremental_lr"],
    update_policy="fixed",
    checkpoint_every=cfg["runtime"]["checkpoint_every_n_sessions"],
    device=cfg["runtime"]["device"],
    seed=DEMO_SEED,
    verbose=True,
)

history = runner.fit_initial(ds["X_train"], ds["y_train"])

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history["train_loss"], lw=1)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title(f"{DEMO_SYMBOL} {DEMO_FSET} seed={DEMO_SEED} — training loss")
ax.set_yscale("log")
plt.tight_layout()
fig.savefig(paths.figures / f"{DEMO_SYMBOL}_{DEMO_FSET}_s{DEMO_SEED}_train_loss.png", dpi=150)
plt.show()

In [ ]:
# Phase 1+2 — walk-forward test
ckpt_path = paths.checkpoint_path(DEMO_SYMBOL, DEMO_SEED)

df_preds = runner.run_test(
    X_test=ds["X_test"],
    y_test=ds["y_test"],
    y_test_raw=ds["y_test_raw"],
    timestamps=ds["ts_test"],
    session_ids=session_ids_test,
    scaler_y=ds["scaler_y"],
    checkpoint_path=ckpt_path,
)

print(df_preds.head())

In [ ]:
# Cost report
cost = runner.cost_report()
print("\nCost report:")
for k, v in cost.items():
    print(f"  {k}: {v}")

## 2. Predictions visualisation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Full test period
axes[0].plot(df_preds.index, df_preds["y_true_raw"],  lw=0.5, label="Actual",    color="steelblue")
axes[0].plot(df_preds.index, df_preds["y_pred_raw"],  lw=0.5, label="Predicted", color="tomato", alpha=0.8)
axes[0].set_title(f"{DEMO_SYMBOL} {DEMO_FSET} — Actual vs Predicted (full test period)")
axes[0].set_ylabel("Price")
axes[0].legend()

# Residuals
residuals = df_preds["y_true_raw"] - df_preds["y_pred_raw"]
axes[1].plot(df_preds.index, residuals, lw=0.4, color="gray")
axes[1].axhline(0, color="red", ls="--", lw=0.8)
axes[1].set_ylabel("Residual (price units)")
axes[1].set_title("Residuals")

plt.tight_layout()
fig.savefig(paths.figures / f"{DEMO_SYMBOL}_{DEMO_FSET}_s{DEMO_SEED}_predictions.png", dpi=150)
plt.show()

In [ ]:
# Zoom in on first 3 trading days
first_3_days = df_preds["session_id"].unique()[:3]
zoom = df_preds[df_preds["session_id"].isin(first_3_days)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(zoom.index, zoom["y_true_raw"],  lw=1.2, label="Actual",    marker="o", ms=2)
ax.plot(zoom.index, zoom["y_pred_raw"],  lw=1.2, label="Predicted", marker="x", ms=2)
ax.set_title(f"{DEMO_SYMBOL} — zoom: first 3 test sessions")
ax.set_ylabel("Price")
ax.legend()
plt.tight_layout()
fig.savefig(paths.figures / f"{DEMO_SYMBOL}_{DEMO_FSET}_s{DEMO_SEED}_zoom.png", dpi=150)
plt.show()

## 3. Quick metric check (single run)

In [ ]:
from src.metrics import evaluate, compare_to_paper, naive_persistence

y_true = df_preds["y_true_raw"].values
y_pred = df_preds["y_pred_raw"].values

result = evaluate(y_true, y_pred, symbol=DEMO_SYMBOL, feature_set=DEMO_FSET, seed=DEMO_SEED)

print(f"\nResults for {DEMO_SYMBOL} {DEMO_FSET} seed={DEMO_SEED}:")
print(f"  MAE  : {result['mae']:.4f}")
print(f"  RMSE : {result['rmse']:.4f}")
print(f"  MAPE : {result['mape']:.4f}")
print(f"  MASE : {result['mase']:.4f}  (< 1 = skill over naive)")
print(f"  DA   : {result['da']:.4f}  (> 0.5 = directional skill)")
print()
print(f"  Paper MAE : 5.617  | Paper RMSE: 8.574  | Paper MAPE: 0.001")
print(f"  (SPX multivariate H.BLSTM, Table 2/3)")

## 4. Save demo predictions

In [ ]:
preds_path = paths.preds_path(DEMO_SYMBOL, DEMO_FSET, DEMO_SEED)
df_preds.to_parquet(preds_path)
print(f"Saved predictions → {preds_path}")

## 5. Full loop — all 3 indices × 2 feature sets × 3 seeds

> **Runtime estimate (CPU):** ~15-40 min per (symbol, feature_set, seed) due to
> end-of-session full retrains. Total ≈ 3–4 hours on CPU; ~30 min on GPU.
>
> Checkpoints are saved every 5 sessions so Colab/Kaggle interruptions are recoverable.

In [ ]:
all_results = []

for idx_cfg in INDICES:
    sym = idx_cfg["symbol"]
    for fset in FEATURE_SETS:
        # Load dataset once per (sym, fset) — shared across seeds
        print(f"\n{'='*60}")
        print(f"Loading dataset: {sym} | {fset}")
        ds = build_dataset(
            symbol=sym, exchange=idx_cfg["exchange"], tz=idx_cfg["tz"],
            feature_set=fset, paths=paths,
            n_bars=N_BARS, lag=LAG, train_ratio=TRAIN_RATIO,
            force_download=False, ema5_period=EMA5_PERIOD,
        )

        df_clean = ds["df_clean"]
        session_ids_test = np.array(
            df_clean.loc[ds["ts_test"], "session_id"].values
        )

        for seed in SEEDS:
            print(f"\n--- {sym} | {fset} | seed={seed} ---")

            preds_path = paths.preds_path(sym, fset, seed)
            if preds_path.exists():
                print(f"  Found cached predictions → {preds_path}")
                df_preds = pd.read_parquet(preds_path)
            else:
                runner = HBLSTMRunner(
                    model_kwargs=MODEL_KWARGS,
                    trainer_kwargs=TRAINER_KWARGS,
                    incremental_lr=cfg["training"]["incremental_lr"],
                    update_policy="fixed",
                    checkpoint_every=cfg["runtime"]["checkpoint_every_n_sessions"],
                    device=cfg["runtime"]["device"],
                    seed=seed,
                    verbose=False,  # reduce verbosity in full loop
                )

                runner.fit_initial(ds["X_train"], ds["y_train"])

                ckpt_path = paths.checkpoint_path(sym, seed)
                df_preds = runner.run_test(
                    X_test=ds["X_test"], y_test=ds["y_test"],
                    y_test_raw=ds["y_test_raw"],
                    timestamps=ds["ts_test"],
                    session_ids=session_ids_test,
                    scaler_y=ds["scaler_y"],
                    checkpoint_path=ckpt_path,
                )

                df_preds.to_parquet(preds_path)
                print(f"  Saved → {preds_path}")

            # Compute metrics
            y_true = df_preds["y_true_raw"].values
            y_pred = df_preds["y_pred_raw"].values
            result = evaluate(y_true, y_pred, symbol=sym, feature_set=fset, seed=seed)
            all_results.append(result)
            print(
                f"  MAE={result['mae']:.3f}  RMSE={result['rmse']:.3f}  "
                f"MAPE={result['mape']:.4f}  MASE={result['mase']:.3f}  DA={result['da']:.3f}"
            )

print(f"\n✓ Full loop complete. {len(all_results)} runs.")

In [ ]:
# Save all results
import json

results_path = paths.results / "hblstm_all_results.json"
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Saved all results → {results_path}")